# Lab 3.3, Builds 1 and 2: Ingest the case files, then filter to the target

The first half of this notebook maps and loads the 120 Cortex case memos. The second
half turns each query's intent into filter clauses and measures what that bought you.

Run the harness cell, then work through the cells marked **YOUR WORK**.


In [ ]:
# Harness. Nothing here is graded.
import json
import os
import pathlib
import sys
import time

sys.path.insert(0, "/opt/ara/lib")  # shared library: ara_metrics, ara_pack, tina

from elasticsearch import Elasticsearch
from ara_metrics import precision_at_k

INDEX = "cortex-cases"
ES = Elasticsearch(os.environ["ES_URL"], api_key=os.environ["ES_API_KEY"], request_timeout=120)
EMBED_ID = os.environ["ARA_EMBED_ID"]
CASE_FILE = pathlib.Path(os.environ.get("ARA_CASE_FILE", "/home/elastic/data/cortex-cases.ndjson"))
TRACES = pathlib.Path("/home/elastic/.traces")
TRACES.mkdir(exist_ok=True)

DEV_QUERIES = json.loads(pathlib.Path("/home/elastic/dev-sets/dev-queries.json").read_text())
MEASURE_SET = DEV_QUERIES[:12]   # the 12 queries you measure before and after
PRACTICE_SET = DEV_QUERIES[12:]  # the rest are yours to iterate on

CASES = [json.loads(line) for line in CASE_FILE.read_text().splitlines() if line.strip()]
print(f"{len(CASES)} case memos on disk, {len(DEV_QUERIES)} dev queries loaded.")
print(f"First memo: {CASES[0]['case_id']}  {CASES[0]['case_type']}  {CASES[0]['risk_tier']}  {CASES[0]['filing_date']}")
print(f"Intent of the first dev query: {MEASURE_SET[0]['intent']}")


## Build 1: the mapping

The mapping is the only retrieval decision in this track you cannot revise later.
Build 2 measures your filtered precision with the mapping frozen.


In [ ]:
# YOUR WORK: complete the mapping, then load the memos.
#
# Replace each None with the type the field needs. The comment beside each one says
# what the field has to support once Build 2 starts filtering.

MAPPING = {
    "properties": {
        "case_id":     {"type": "keyword"},
        "case_type":   {"type": None},   # selected on exactly, and counted per value
        "risk_tier":   {"type": None},   # selected on exactly, one of three values
        "filing_date": {"type": None},   # date windows, both ends
        "title":       {"type": None},   # read by people, ranked lexically, never filtered
        "body_text":   {"type": None},   # the lexical half of hybrid retrieval
        "body":        {"type": "semantic_text", "inference_id": EMBED_ID},
    }
}

# No number_of_shards, no number_of_replicas, no index_options on semantic_text:
# this platform rejects all three at request time with no readable message.
ES.indices.delete(index=INDEX, ignore_unavailable=True)
ES.indices.create(index=INDEX, body={"mappings": MAPPING})
print(f"{INDEX} created.")

BATCH = 20
for start in range(0, len(CASES), BATCH):
    operations = []
    for memo in CASES[start:start + BATCH]:
        operations.append({"index": {"_index": INDEX, "_id": memo["case_id"]}})
        operations.append({
            "case_id": memo["case_id"],
            "case_type": memo["case_type"],
            "risk_tier": memo["risk_tier"],
            "filing_date": memo["filing_date"],
            "title": memo["title"],
            "body_text": memo["body"],   # the twin: same text, lexical field
            "body": memo["body"],        # the twin: same text, semantic field
        })
    response = ES.bulk(operations=operations, refresh=False)
    if response.get("errors"):
        failed = [item for item in response["items"] if list(item.values())[0].get("error")]
        raise RuntimeError(f"{len(failed)} documents failed, first: {failed[0]}")
    print(f"  loaded {min(start + BATCH, len(CASES))} of {len(CASES)}")

print("Bulk load accepted.")


In [ ]:
# Verify the load. Semantic inference runs during ingest, so give it a moment to drain.
for _ in range(60):
    pending = [t for t in ES.cat.tasks(detailed=True, format="json")
               if "inference" in str(t.get("description", "")).lower()]
    if not pending:
        break
    time.sleep(5)

ES.indices.refresh(index=INDEX)
count = ES.count(index=INDEX)["count"]
print(f"documents: {count}")

aggregation = ES.search(
    index=INDEX,
    size=0,
    aggs={"types": {"terms": {"field": "case_type", "size": 10}}},
)
buckets = aggregation["aggregations"]["types"]["buckets"]
print(f"case_type buckets: {len(buckets)}")
for bucket in buckets:
    print(f"  {bucket['key']}: {bucket['doc_count']}")

if count != 120 or len(buckets) != 5:
    print("\nNot there yet. 120 documents and 5 buckets is the target.")
    print("Fragments of words instead of whole values means the field is analyzed.")
else:
    print("\nReady. Select Check, then continue with Build 2 below.")


## Build 2: filters from intent

Unfiltered retrieval over these memos returns roughly half the right files, because
every memo borrows a second case type's vocabulary. Your target is precision at 10
of at least 0.80 on held-out queries.

The grader executes the cells that define `build_filters` and `build_retriever`, in
order, and nothing else. Keep both self-contained.


In [ ]:
# YOUR WORK: turn each query's intent into filter clauses.
#
# The intent the harness extracts has four keys. A key set to None means the query
# put no constraint on that field, so no clause belongs in the list for it.
#
#   {"case_type": "sanctions", "risk_tier": ["medium", "high"],
#    "date_from": "2024-01-01", "date_to": None}


def build_filters(intent: dict) -> list:
    """Return a list of Elasticsearch filter clauses for this intent."""
    clauses = []
    # TODO: one clause per constraint the intent names.
    #   a single value and a list of values are different clauses
    #   a window with one end set is still a range with one bound
    return clauses


def build_retriever(query_text: str, filters: list) -> dict:
    """Hybrid retrieval over cortex-cases with your filter clauses applied."""
    clauses = list(filters or [])

    def leaf(inner: dict) -> dict:
        # TODO: when clauses is non-empty, wrap `inner` so the clauses narrow this
        # half of the fusion as well. Both halves need them, or the unfiltered half
        # contributes contaminated documents into the fusion window.
        return {"standard": {"query": inner}}

    return {
        "retriever": {
            "rrf": {
                "retrievers": [
                    leaf({"match": {"body_text": query_text}}),
                    leaf({"semantic": {"field": "body", "query": query_text}}),
                ],
                "rank_window_size": 50,
                "rank_constant": 60,
            }
        },
        "size": 10,
    }


In [ ]:
# Measure before and after on the 12 dev queries.


def run(queries, filtered):
    results = []
    for query in queries:
        clauses = build_filters(query["intent"]) if filtered else []
        body = build_retriever(query["query_text"], clauses)
        response = ES.search(index=INDEX, body=body)
        results.append({
            "query_id": query["query_id"],
            "retrieved_ids": [hit.get("_source", {}).get("doc_id") or hit["_id"]
                              for hit in response["hits"]["hits"]],
            "latency_ms": 0.0,
        })
    return results


baseline_results = run(MEASURE_SET, filtered=False)
filtered_results = run(MEASURE_SET, filtered=True)

baseline = precision_at_k(baseline_results, MEASURE_SET, k=10)
filtered = precision_at_k(filtered_results, MEASURE_SET, k=10)

print(f"unfiltered precision@10: {baseline:.3f}")
print(f"filtered   precision@10: {filtered:.3f}")
print(f"gain: {filtered - baseline:+.3f}\n")

by_id = {query["query_id"]: query for query in MEASURE_SET}
MISSES = []
for result in filtered_results:
    query = by_id[result["query_id"]]
    gold = set(query["relevant_ids"])
    unexpected = [doc for doc in result["retrieved_ids"][:10] if doc not in gold]
    if unexpected:
        MISSES.append({
            "query_id": query["query_id"],
            "intent": query["intent"],
            "filters": build_filters(query["intent"]),
            "unexpected_ids": unexpected,
        })

print(f"{len(MISSES)} of {len(MEASURE_SET)} queries still return a document outside the gold set.")
for miss in MISSES[:4]:
    print(f"  {miss['query_id']}  intent={miss['intent']}")
    print(f"    clauses: {miss['filters']}")
    print(f"    unexpected: {miss['unexpected_ids'][:3]}")


In [ ]:
# Save the results file the grader reads.
payload = {
    "baseline_precision_at_10": round(baseline, 3),
    "filtered_precision_at_10": round(filtered, 3),
    "dev_query_count": len(MEASURE_SET),
    "misses": MISSES,
}
(TRACES / "filter-results.json").write_text(json.dumps(payload, indent=2))
print(f"filter-results.json written: {payload['baseline_precision_at_10']} "
      f"to {payload['filtered_precision_at_10']} over {payload['dev_query_count']} queries.")
print("Select Check.")
